<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/04_multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Multimodal experiments

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation.

This notebook trains text-augmented variants of UNetFormer (UNetFormer + frozen CLIP via cross-attention) and addresses the project's three research questions:

- **RQ1**: Does textual information improve segmentation over the vision-only baseline?
- **Sub-question 1 (caption-source ablation)**: Which caption type (hybrid, text-only, vision-only) gives the largest improvement?
- **Sub-question 2 (auxiliary supervision)**: Does adding a KL-divergence loss between predicted and ground-truth class distributions help on top of the best caption type?

The vision-only baseline (UNetFormer, no text) is established in `03_baselines.ipynb`. Multimodal experiments here use the same training setup (30 epochs, AdamW lr=6e-4, weighted CE + 0.4 × aux head loss, batch size 8, no augmentation) and only differ in the addition of text fusion.

Sections:
1. Setup
2. Dataset and DataLoaders (with caption)
3. UNetFormer architecture
4. Multimodal architecture (CLIP + cross-attention)
5. Training infrastructure
6. Caption-source ablation (RQ1 + Sub1)
7. Auxiliary KL-divergence loss (Sub2)
8. Test set evaluation

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies. open_clip_torch provides the frozen CLIP text encoder.
!pip install transformers timm einops wandb open_clip_torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil, os

PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time
import json
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [6]:
# Weights & Biases setup for experiment tracking
import wandb
wandb.login()

WANDB_PROJECT = 'multimodal-rs-segmentation'

In [7]:
# Class definitions
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# 5 caption variants from the dataset
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

In [8]:
# Load split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Class weights via median frequency balancing (computed from training split)
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


## 2. Dataset and DataLoaders

The Dataset returns (image, mask, caption) tuples. The caption column is a constructor argument so the same class works for any of the 5 caption variants.

In [9]:
# Dataset class — reads pre-converted class-index masks and a caption column
class RSMultiModalDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [10]:
BATCH_SIZE  = 8
NUM_WORKERS = 4

def make_loaders(caption_col):
    """Build train/val/test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

# Sanity check with the first caption variant
train_loader, val_loader, test_loader = make_loaders(CAPTION_COLS[0])
print(f'Caption column: {CAPTION_COLS[0]}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}, captions[0]: "{captions[0][:80]}..."')

Caption column: hybrid_gemma3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256]), captions[0]: "The image depicts a landscape dominated by extensive grasslands (81%) interspers..."


## 3. UNetFormer architecture

Same UNetFormer as in `03_baselines.ipynb` — CNN encoder (`swsl_resnet18`) with Global-Local Transformer Block decoder, Feature Refinement Head, and auxiliary head for deep supervision. Re-defined here so this notebook is self-contained.

Source: https://github.com/WangLibo1995/GeoSeg

In [11]:
# Building blocks (Conv + BN + ReLU variants)
from einops import rearrange
from timm.models.layers import DropPath, trunc_normal_
import timm


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [12]:
# MLP and Global-Local Attention (window self-attention + local conv branch)
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [13]:
# Transformer Block (GLTB)
class Block(nn.Module):
    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [14]:
# Decoder components: weighted feature fusion (WF), feature refinement head, aux head
class WF(nn.Module):
    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

The multimodal model extends UNetFormer with three additions:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Text-Visual Cross-Attention**: visual features attend to the text embedding via a gated residual. The gate parameter starts at zero, so text influence ramps up gradually during training instead of disrupting the pretrained backbone.
3. **TextAugmentedDecoder**: same as the UNetFormer decoder, but with a cross-attention module after each Global-Local Transformer Block.

Only the cross-attention modules and the decoder are trained. The CNN encoder (`swsl_resnet18`) and CLIP text encoder are both frozen — same setup as PoC.

In [15]:
import open_clip


class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower — we only use the text encoder.
        # Saves ~88M frozen params from GPU memory.
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Verify
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [16]:
class TextVisualCrossAttention(nn.Module):
    """Visual features attend to text embedding via gated residual.

    The gate starts at 0, meaning text initially has no influence. As training
    progresses, the gate learns to weight the text contribution, allowing the
    backbone to retain its pretrained features at the start.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text embedding into the visual feature space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        # Gated residual — starts at 0, learns to scale text contribution
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [17]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder + cross-attention after each GLTB."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Text-visual cross-attention at each decoder stage
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [18]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer + frozen CLIP text encoder + cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [19]:
# Quick sanity check: instantiate, count parameters, forward pass in both modes
test_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters   : {frozen_params / 1e6:.2f}M (CLIP text encoder)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])

    test_model.eval()
    out_eval = test_model(sample_imgs, sample_caps)
    print(f'\nEval  mode: input {sample_imgs.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_imgs, sample_caps)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters    : 75.37M
Trainable parameters: 11.94M
Frozen parameters   : 63.43M (CLIP text encoder)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])


## 5. Training infrastructure

Training and evaluation functions for the multimodal model. Same hyperparameters as the UNetFormer baseline (30 epochs, AdamW lr=6e-4, weight decay 0.01, CosineAnnealingLR, weighted CE loss + 0.4 × auxiliary head loss).

The training function accepts an optional `kl_weight` argument. When non-zero, an additional KL-divergence loss is added between the predicted class distribution (mean-pooled over spatial dimensions) and the ground-truth class distribution from the CSV. This is used in Section 7 for Sub-question 2.

Each training run is logged to Weights & Biases under the project `multimodal-rs-segmentation`.

In [20]:
# Training hyperparameters (same as UNetFormer baseline)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4

In [21]:
def compute_metrics(pred, target, num_classes=NUM_CLASSES):
    """Per-image metrics. pred and target: tensors of shape (H, W) with class indices."""
    ious = []
    for cls in range(num_classes):
        pred_cls   = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union        = (pred_cls | target_cls).sum().item()
        ious.append(float('nan') if union == 0 else intersection / union)
    valid_ious = [x for x in ious if not np.isnan(x)]
    miou = np.mean(valid_ious) if valid_ious else 0.0
    oa = (pred == target).sum().item() / target.numel()
    return ious, miou, oa

In [22]:
def train_one_epoch(model, loader, optimizer, criterion, device, kl_weight=0.0):
    """One training epoch.

    If kl_weight > 0, adds KL-divergence loss between predicted class
    distribution (spatial mean of softmax) and ground-truth distribution
    (from per-image class percentages).
    """
    model.train()
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs, list(captions))

        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)

        if kl_weight > 0:
            # Predicted class distribution: spatial mean of softmax probabilities
            pred_dist = F.softmax(logits_main, dim=1).mean(dim=[2, 3])  # (B, num_classes)
            # Ground-truth distribution: pixel proportions per class in the mask
            gt_dist = torch.zeros_like(pred_dist)
            for c in range(NUM_CLASSES):
                gt_dist[:, c] = (masks == c).float().mean(dim=[1, 2])
            # KL(pred || gt) — F.kl_div expects log-probabilities for the input
            kl_loss = F.kl_div(
                torch.log(pred_dist + 1e-8), gt_dist + 1e-8,
                reduction='batchmean',
            )
            loss = loss + kl_weight * kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns per-class IoU list, mIoU, overall accuracy, average loss."""
    model.eval()
    all_ious = [[] for _ in range(num_classes)]
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1).cpu()
        masks_cpu = masks.cpu()
        for pred, mask in zip(preds, masks_cpu):
            ious, _, _ = compute_metrics(pred, mask, num_classes)
            for c in range(num_classes):
                if not np.isnan(ious[c]):
                    all_ious[c].append(ious[c])
            total_correct += (pred == mask).sum().item()
            total_pixels  += mask.numel()
    class_ious = [np.mean(all_ious[c]) if all_ious[c] else float('nan') for c in range(num_classes)]
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [23]:
def train_multimodal(caption_col, num_epochs=NUM_EPOCHS, lr=LR, kl_weight=0.0,
                     run_suffix=None):
    """Train UNetFormerCLIP for a given caption column.

    Args:
        caption_col: which caption column from CAPTION_COLS to use as text input.
        num_epochs: number of epochs to train.
        lr: initial learning rate.
        kl_weight: weight on the KL-divergence auxiliary loss (0 = disabled).
        run_suffix: optional string appended to the run name (e.g. '+kl').

    Returns:
        history dict, best_miou, save_path.
    """
    # Build identifiers
    base_name = caption_col.replace('-', '_').replace('/', '_')
    run_name = f'multimodal_{base_name}'
    if run_suffix:
        run_name += f'_{run_suffix}'
    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    # Loaders for this caption variant
    train_loader, val_loader, _ = make_loaders(caption_col)

    # Model
    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

    # Optimizer over only the trainable params (skips frozen CLIP)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # wandb run
    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            'model':           'UNetFormerCLIP',
            'caption_col':     caption_col,
            'kl_weight':       kl_weight,
            'num_epochs':      num_epochs,
            'lr':              lr,
            'weight_decay':    WEIGHT_DECAY,
            'batch_size':      BATCH_SIZE,
            'aux_loss_weight': AUX_LOSS_WEIGHT,
            'seed':            SEED,
        },
        reinit=True,
    )

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device,
                                     kl_weight=kl_weight)
        val_class_ious, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()

    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path


def save_history(history, name):
    """Persist training history to disk."""
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')

## 6. Caption-source ablation (RQ1 + Sub-question 1)

Train UNetFormerCLIP separately for each of the 5 caption variants:

- **Hybrid** (`hybrid_gemma3-4b`, `hybrid_qwen3-vl-8b`): generated from both vision and ground-truth class percentages.
- **Text-only** (`text_qwen3-4b`): generated from class percentages only.
- **Vision-only** (`vision_gemma3-4b`, `vision_qwen3-vl-8b`): generated from the image alone.

Each run uses the same hyperparameters as the UNetFormer baseline. All five runs are tracked in WANDB under the `multimodal-rs-segmentation` project.

The vision-only baseline (UNetFormer without text) from `03_baselines.ipynb` provides the reference point for **RQ1** (does text help at all?). Comparing across the 5 variants here addresses **Sub-question 1** (which caption source helps most?).

⚠ Each run takes ~25–30 minutes on Colab A100. Total ~2.5 hours for all 5.

In [24]:
# Run 1/5: hybrid_gemma3-4b
hist_hyb_gemma, miou_hyb_gemma, ckpt_hyb_gemma = train_multimodal('hybrid_gemma3-4b')
save_history(hist_hyb_gemma, 'multimodal_hybrid_gemma3_4b')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training multimodal_hybrid_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1140   0.4540   0.2449   0.7429   67.1s   BEST
    2   0.7469   0.4429   0.2332   0.7269   60.6s       
    3   0.6348   0.4138   0.2698   0.7746   61.6s   BEST
    4   0.6075   0.3825   0.2856   0.7881   64.0s   BEST
    5   0.5456   0.3990   0.2793   0.7695   62.2s       
    6   0.5247   0.3409   0.2696   0.7754   61.6s       
    7   0.4994   0.3071   0.2813   0.7975   61.4s       
    8   0.4897   0.3073   0.3104   0.8229   61.7s   BEST
    9   0.4461   0.2877   0.3096   0.8010   61.0s       
   10   0.4365   0.2856   0.2972   0.8171   61.4s       
   11   0.4138   0.2781   0.3012   0.8163   61.9s       
   12   0.4108   0.2745   0.3191   0.8363   63.1s   BEST
   13   0.3969   0.2679   0.3425   0.8336   62.9s   BEST
   14   0.3806   0.2776   0.3476   0.8545   62.2s   BEST
   15   0.3587   0.2447   0.3535   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▁▃▄▂▃▃▆▄▃▄▅▄▇▆▇▇▆▇▅▅▆▆▇█▇▇██▇
val_iou/Built-up,▃▁▃▅▄▂▂▄▄▂▃▄▅▃▅█▆▆▇▅▅▆▆▆▅▆▇▆▇▇
val_iou/Crop,▁▁▁▃▂▂▄▂▅▅▅▅▅▅▆▆▆▇▇▇▇▇████████
val_iou/Grass,▂▁▃▄▄▄▄▅▄▅▅▆▆▇▆▇▅▇▇▇▇▇████████
val_iou/Shrub,▁▃▄▃▃▃▄▂▂▅▄▄▇▃▄▄▄▅▅▇▅▄▅▅▅▅▅▆█▅
val_iou/Tree,▁▃▃▂▁▃▅▅▃▄▄▆▅▆▆▆▇▇▇█▇▇▇███████
val_iou/Water,▂▁▃▃▆▂▁▅▇▃▃▃█▇▇▇▆▃▆▆▃▆▄▅▆▅▆▆▆▆
+3,...



Best validation mIoU: 0.3901
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_history.json


In [25]:
# Run 2/5: hybrid_qwen3-vl-8b
hist_hyb_qwen, miou_hyb_qwen, ckpt_hyb_qwen = train_multimodal('hybrid_qwen3-vl-8b')
save_history(hist_hyb_qwen, 'multimodal_hybrid_qwen3_vl_8b')

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


Training multimodal_hybrid_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1072   0.4563   0.2547   0.7502   67.2s   BEST
    2   0.7061   0.4316   0.2494   0.7302   61.2s       
    3   0.6341   0.3612   0.2862   0.7738   62.6s   BEST
    4   0.5722   0.4586   0.2585   0.7790   61.2s       
    5   0.5463   0.3579   0.2660   0.7779   63.0s       
    6   0.5181   0.3501   0.2818   0.8041   61.3s       
    7   0.4908   0.3650   0.2919   0.8033   61.9s   BEST
    8   0.4619   0.3057   0.2967   0.8066   62.1s   BEST
    9   0.4559   0.4502   0.2882   0.7813   62.3s       
   10   0.4297   0.2813   0.3045   0.8095   62.1s   BEST
   11   0.4150   0.2710   0.3182   0.8281   62.5s   BEST
   12   0.4192   0.2994   0.3216   0.8485   62.2s   BEST
   13   0.4052   0.2630   0.3472   0.8477   62.5s   BEST
   14   0.3659   0.2587   0.3289   0.8406   61.5s       
   15   0.3698   0.2482   0.3489 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▃▃▂▅▄▄▅▃▄▅▆▅▅▆▆▇▆▆█▇▆█▇▇█▇██
val_iou/Built-up,▁▁▃▃▁▄▃▃▇▄▄▄▆▃▅▇█▅▇▅▆▆▆█▇▇█▆▆▇
val_iou/Crop,▂▂▄▃▄▄▃▅▁▅▆▄▆▆▆▆▆▇▇▇▇█████▇███
val_iou/Grass,▁▃▃▄▄▄▅▄▂▅▅▆▆▆▆▇▆▇▆▇▇▇██▇█████
val_iou/Shrub,▁▁▂▃▂▂▅▅▆▄▄▅▃▄▅▄▅▅▃▅▅▅▇▇▆█▅▇▇▆
val_iou/Tree,▄▂▄▁▄▄▅▅▄▅▆▇▇▇▇▇▆█▇█▇█████████
val_iou/Water,▅▃▆▂▂▁▃▄▅▄▅▄▇▅▇▅▂▆█▄▅█▆▇▆▇▆▇▇▆
+3,...



Best validation mIoU: 0.3803
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_qwen3_vl_8b_history.json


In [26]:
# Run 3/5: text_qwen3-4b
hist_text_qwen, miou_text_qwen, ckpt_text_qwen = train_multimodal('text_qwen3-4b')
save_history(hist_text_qwen, 'multimodal_text_qwen3_4b')

Training multimodal_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1532   0.4987   0.2379   0.7507   63.3s   BEST
    2   0.7512   0.4597   0.2323   0.7491   61.6s       
    3   0.6764   0.4587   0.2779   0.7942   62.0s   BEST
    4   0.5986   0.4042   0.2803   0.7781   63.1s   BEST
    5   0.5641   0.3458   0.2704   0.7960   62.0s       
    6   0.5395   0.3761   0.2742   0.7856   62.3s       
    7   0.5191   0.3596   0.2632   0.7883   61.6s       
    8   0.4936   0.3292   0.2767   0.8020   61.8s       
    9   0.4783   0.3041   0.3155   0.8291   63.2s   BEST
   10   0.4483   0.3790   0.2901   0.8222   61.4s       
   11   0.4381   0.3100   0.3100   0.8289   61.3s       
   12   0.4154   0.2802   0.3232   0.8415   61.5s   BEST
   13   0.3962   0.2788   0.3000   0.8202   61.2s       
   14   0.3908   0.3058   0.3149   0.8285   61.7s       
   15   0.3711   0.2524   0.3414   0.8

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▂▄▂▂▅▁▂▅▅▆▄▃▃▆▇▆▆▇▇▆▇▇▇▇▆▇█▇▇
val_iou/Built-up,▃▁▅▆▅▆▃▃▆▆▅▅▅▆▆▆█▆▆▇▆▆▆█▇▆▇█▇█
val_iou/Crop,▂▂▁▁▃▅▃▄▅▄▄▄▅▆▆▆▆▇▇▇▇█████████
val_iou/Grass,▂▁▅▃▄▃▄▅▅▆▆▇▅▅▆▆▇▇▇█▇█████████
val_iou/Shrub,▁▂▃▃▃▂▅▄▅▆▅▆▄█▆▂▇▅▄▆▃▇▇▇▇█▇█▅▆
val_iou/Tree,▁▃▃▅▄▄▅▅▆▄▄▆▆▆▇▆▄▇▇▇▇█▇███████
val_iou/Water,▄▃▅▆▃▂▃▃▅▁▅▆▄▄▆█▁██▆▆▇▇▇█▇▇███
+3,...



Best validation mIoU: 0.3822
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_text_qwen3_4b_history.json


In [27]:
# Run 4/5: vision_gemma3-4b
hist_vis_gemma, miou_vis_gemma, ckpt_vis_gemma = train_multimodal('vision_gemma3-4b')
save_history(hist_vis_gemma, 'multimodal_vision_gemma3_4b')

Training multimodal_vision_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3204   0.7023   0.2117   0.6574   63.1s   BEST
    2   1.0131   0.6044   0.2267   0.6600   62.5s   BEST
    3   0.9101   0.5911   0.2455   0.6753   62.3s   BEST
    4   0.8207   0.5683   0.2159   0.6977   61.2s       
    5   0.7788   0.4464   0.2844   0.7774   61.9s   BEST
    6   0.7430   0.5620   0.2558   0.7309   61.6s       
    7   0.7258   0.4621   0.2401   0.7020   62.4s       
    8   0.6715   0.4056   0.2791   0.7566   62.3s       
    9   0.6558   0.4564   0.2610   0.7531   61.6s       
   10   0.6117   0.4104   0.2669   0.7644   62.0s       
   11   0.6095   0.4371   0.2465   0.7342   61.4s       
   12   0.5624   0.3483   0.3000   0.7897   62.3s   BEST
   13   0.5480   0.3572   0.3036   0.7867   62.5s   BEST
   14   0.5366   0.3947   0.2933   0.7893   61.4s       
   15   0.5127   0.3693   0.3272   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▁▂▂▅▄▂▄▃▄▄▅▅▂▅▄▅▅▅▅▆▇▇█▆▇▇▇▇█
val_iou/Built-up,▁▄▆▃▆▆▄▆▇▄▄▆▇▆▆██▅▆▆▆▇▇▇▆▇▇▇▇▇
val_iou/Crop,▂▁▁▄▅▄▄▄▄▆▅▆▆▆▆▄▆▇▆▇▇██▇██████
val_iou/Grass,▁▂▂▂▅▃▃▅▅▅▃▅▅▆▆▅▆▇▇▆▇▇█▇▇█████
val_iou/Shrub,▁▃▂▁▄▃▂▃▅▂▂▃▄▇▆▃▆▅▅▅▇▆█▅▇█▇▇▇█
val_iou/Tree,▂▂▄▂▄▄▂▄▃▄▃▆▆▅▆▁▆▇▆▇▇▇████████
val_iou/Water,▅▄▅▁▄▃▄▅▁▃▂▅▅▄█▆▆█▆█▆▅▆▇▆▆▅▆▆▇
+3,...



Best validation mIoU: 0.3568
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_vision_gemma3_4b_history.json


In [28]:
# Run 5/5: vision_qwen3-vl-8b
hist_vis_qwen, miou_vis_qwen, ckpt_vis_qwen = train_multimodal('vision_qwen3-vl-8b')
save_history(hist_vis_qwen, 'multimodal_vision_qwen3_vl_8b')

Training multimodal_vision_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3081   0.8843   0.1658   0.6254   64.9s   BEST
    2   0.9931   0.6162   0.2225   0.6769   63.4s   BEST
    3   0.8925   0.5531   0.2661   0.7609   62.8s   BEST
    4   0.8272   0.5962   0.2314   0.6534   62.8s       
    5   0.7685   0.6008   0.1889   0.6224   63.1s       
    6   0.7624   0.5236   0.2180   0.6876   63.0s       
    7   0.7089   0.4253   0.2687   0.7388   63.1s   BEST
    8   0.6726   0.4299   0.2592   0.7365   62.4s       
    9   0.6432   0.4158   0.2731   0.7371   63.7s   BEST
   10   0.6245   0.4127   0.2625   0.7547   63.7s       
   11   0.5731   0.3915   0.2990   0.7932   63.3s   BEST
   12   0.5796   0.3990   0.2882   0.7762   63.1s       
   13   0.5396   0.3787   0.3029   0.8012   63.3s   BEST
   14   0.5559   0.4738   0.2953   0.7518   62.8s       
   15   0.5077   0.3870   0.3025 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▃▃▄▃▃▄▄▄▄▅▅▅▅▄█▄▆▇▇▇▆▆█▇▆▇█▇█
val_iou/Built-up,▁▅▂▆▂▄▄▅▆▇▃▆▄▆▅█▄▆▇▇█▆▆▆▇▆▇▇▆▆
val_iou/Crop,▁▂▄▃▂▃▅▅▄▃▆▅▆▅▆▇▆▆▇▇▆█▇█▇██▇██
val_iou/Grass,▁▃▅▁▂▄▅▄▄▄▅▅▆▄▆▆▆▇█▇▇▇███▇▇███
val_iou/Shrub,▁▄▅▃▂▄▄▃▄▄▅▃▅▄▇▄▄▇█▆▅▇▇▇▆▆▇█▇█
val_iou/Tree,▂▃▄▄▁▃▅▅▅▅▇▆▆▇▇▇▅▇█▇▇█████████
val_iou/Water,▁▃▇▅▂▂▆▅▆▄▇▆▇█▆▇█▇█▆▇▆▇▆▆▆▆▇▇▇
+3,...



Best validation mIoU: 0.3510
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_vision_qwen3_vl_8b_history.json


In [29]:
# Summary of caption-source ablation
caption_results = {
    'hybrid_gemma3-4b':    miou_hyb_gemma,
    'hybrid_qwen3-vl-8b':  miou_hyb_qwen,
    'text_qwen3-4b':       miou_text_qwen,
    'vision_gemma3-4b':    miou_vis_gemma,
    'vision_qwen3-vl-8b':  miou_vis_qwen,
}

print(f"{'Caption variant':<25} {'Best val mIoU':>15}")
print('-' * 42)
for cap, miou in sorted(caption_results.items(), key=lambda x: -x[1]):
    print(f'{cap:<25} {miou:>15.4f}')

best_caption = max(caption_results, key=caption_results.get)
print(f'\nBest caption variant: {best_caption} (val mIoU = {caption_results[best_caption]:.4f})')

Caption variant             Best val mIoU
------------------------------------------
hybrid_gemma3-4b                   0.3901
text_qwen3-4b                      0.3822
hybrid_qwen3-vl-8b                 0.3803
vision_gemma3-4b                   0.3568
vision_qwen3-vl-8b                 0.3510

Best caption variant: hybrid_gemma3-4b (val mIoU = 0.3901)


## 7. Test set evaluation — multimodal vs baseline

Evaluate each trained multimodal model on the held-out test set and compare against the vision-only UNetFormer baseline from `03_baselines.ipynb`. Test mIoU is the final reported number for each configuration; the variant with the highest test mIoU is selected for the KL-divergence ablation in the next section.

In [30]:
# Load the vision-only baseline result for the comparison table
with open(RESULTS_DIR / 'baselines_results.json') as f:
    baseline_data = json.load(f)

unet_baseline_test_miou  = baseline_data['unetformer']['test_miou']
unet_baseline_test_oa    = baseline_data['unetformer']['test_oa']
unet_baseline_class_ious = baseline_data['unetformer']['class_ious']

print(f'Vision-only UNetFormer baseline:')
print(f'  test mIoU: {unet_baseline_test_miou:.4f}')
print(f'  test OA  : {unet_baseline_test_oa:.4f}')

Vision-only UNetFormer baseline:
  test mIoU: 0.3509
  test OA  : 0.8522


In [31]:
def evaluate_checkpoint(caption_col, ckpt_path):
    """Reload a trained UNetFormerCLIP checkpoint and evaluate on the test set."""
    _, _, test_loader = make_loaders(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    class_ious, miou, oa, loss = evaluate(model, test_loader, criterion, device)

    # Free memory before next eval
    del model
    torch.cuda.empty_cache()

    return class_ious, miou, oa, loss

In [32]:
multimodal_runs = [
    ('hybrid_gemma3-4b',   ckpt_hyb_gemma),
    ('hybrid_qwen3-vl-8b', ckpt_hyb_qwen),
    ('text_qwen3-4b',      ckpt_text_qwen),
    ('vision_gemma3-4b',   ckpt_vis_gemma),
    ('vision_qwen3-vl-8b', ckpt_vis_qwen),
]

multimodal_test_results = {}
for caption_col, ckpt_path in multimodal_runs:
    print(f'Evaluating {caption_col}...')
    class_ious, miou, oa, loss = evaluate_checkpoint(caption_col, ckpt_path)
    multimodal_test_results[caption_col] = {
        'class_ious': class_ious,
        'miou': miou,
        'oa': oa,
        'loss': loss,
    }
    print(f'  test mIoU: {miou:.4f}, OA: {oa:.4f}')

Evaluating hybrid_gemma3-4b...
  test mIoU: 0.3910, OA: 0.8871
Evaluating hybrid_qwen3-vl-8b...
  test mIoU: 0.3824, OA: 0.8813
Evaluating text_qwen3-4b...
  test mIoU: 0.3853, OA: 0.8854
Evaluating vision_gemma3-4b...
  test mIoU: 0.3600, OA: 0.8602
Evaluating vision_qwen3-vl-8b...
  test mIoU: 0.3521, OA: 0.8550


In [33]:
# Build comparison table: per-class IoU, mIoU, OA across baseline + 5 multimodal variants
header_runs = ['vision-only'] + [c.replace('_', '\n') for c, _ in multimodal_runs]
header = f"{'Class':<11}" + ''.join(f'{r:>14}' for r in [
    'vision-only',
    'hyb_gemma',
    'hyb_qwen',
    'text_qwen',
    'vis_gemma',
    'vis_qwen',
])
print(header)
print('-' * len(header))

for i, name in enumerate(CLASS_NAMES):
    row = f'{name:<11}'
    # baseline
    bv = unet_baseline_class_ious[i]
    row += f'{(bv if bv is not None else 0):>14.4f}' if bv is not None else f'{"N/A":>14}'
    # multimodal
    for caption_col, _ in multimodal_runs:
        v = multimodal_test_results[caption_col]['class_ious'][i]
        row += f'{v:>14.4f}' if not np.isnan(v) else f'{"N/A":>14}'
    print(row)

print('-' * len(header))
miou_row = f'{"mIoU":<11}{unet_baseline_test_miou:>14.4f}'
for caption_col, _ in multimodal_runs:
    miou_row += f'{multimodal_test_results[caption_col]["miou"]:>14.4f}'
print(miou_row)

oa_row = f'{"OA":<11}{unet_baseline_test_oa:>14.4f}'
for caption_col, _ in multimodal_runs:
    oa_row += f'{multimodal_test_results[caption_col]["oa"]:>14.4f}'
print(oa_row)

Class         vision-only     hyb_gemma      hyb_qwen     text_qwen     vis_gemma      vis_qwen
-----------------------------------------------------------------------------------------------
Tree               0.5963        0.6148        0.6057        0.6094        0.5975        0.5884
Shrub              0.0848        0.1238        0.0996        0.1066        0.0974        0.0977
Grass              0.5842        0.6169        0.6043        0.6104        0.5945        0.5830
Crop               0.3948        0.4595        0.4491        0.4527        0.4011        0.3914
Built-up           0.2225        0.2718        0.2669        0.2696        0.2231        0.2321
Barren             0.2640        0.3171        0.3158        0.3122        0.2896        0.2672
Water              0.3098        0.3329        0.3350        0.3359        0.3165        0.3050
-----------------------------------------------------------------------------------------------
mIoU               0.3509        0.3910 

In [34]:
# Δ vs baseline summary
print(f"\n{'Caption variant':<25} {'Test mIoU':>10} {'Δ vs baseline':>16} {'Relative %':>12}")
print('-' * 65)
print(f'{"vision-only (baseline)":<25} {unet_baseline_test_miou:>10.4f} {"--":>16} {"--":>12}')
for caption_col, _ in multimodal_runs:
    miou = multimodal_test_results[caption_col]['miou']
    delta = miou - unet_baseline_test_miou
    rel = delta / unet_baseline_test_miou * 100
    print(f'{caption_col:<25} {miou:>10.4f} {delta:>+16.4f} {rel:>+11.1f}%')

# Best by test mIoU
best_caption_test = max(multimodal_test_results, key=lambda k: multimodal_test_results[k]['miou'])
best_test_miou = multimodal_test_results[best_caption_test]['miou']
print(f'\nBest caption variant on test set: {best_caption_test} (mIoU = {best_test_miou:.4f})')


Caption variant            Test mIoU    Δ vs baseline   Relative %
-----------------------------------------------------------------
vision-only (baseline)        0.3509               --           --
hybrid_gemma3-4b              0.3910          +0.0400       +11.4%
hybrid_qwen3-vl-8b            0.3824          +0.0314        +9.0%
text_qwen3-4b                 0.3853          +0.0344        +9.8%
vision_gemma3-4b              0.3600          +0.0090        +2.6%
vision_qwen3-vl-8b            0.3521          +0.0012        +0.3%

Best caption variant on test set: hybrid_gemma3-4b (mIoU = 0.3910)


In [36]:
# Detailed per-class comparison: vision-only baseline vs best multimodal variant
best_results = multimodal_test_results[best_caption_test]

print(f"Per-class comparison: vision-only vs best multimodal ({best_caption_test})\n")
print(f"{'Class':<12} {'Baseline':>10} {'Multimodal':>12} {'Δ':>10} {'Rel %':>10}")
print('-' * 56)

for i, name in enumerate(CLASS_NAMES):
    base = unet_baseline_class_ious[i]
    mm   = best_results['class_ious'][i]
    if base is None or np.isnan(mm):
        print(f'{name:<12} {"N/A":>10} {"N/A":>12} {"":>10} {"":>10}')
        continue
    delta = mm - base
    rel = (delta / base * 100) if base > 0 else float('nan')
    print(f'{name:<12} {base:>10.4f} {mm:>12.4f} {delta:>+10.4f} {rel:>+9.1f}%')

print('-' * 56)
print(f'{"mIoU":<12} {unet_baseline_test_miou:>10.4f} {best_results["miou"]:>12.4f} '
      f'{best_results["miou"] - unet_baseline_test_miou:>+10.4f} '
      f'{(best_results["miou"] - unet_baseline_test_miou) / unet_baseline_test_miou * 100:>+9.1f}%')
print(f'{"OA":<12} {unet_baseline_test_oa:>10.4f} {best_results["oa"]:>12.4f} '
      f'{best_results["oa"] - unet_baseline_test_oa:>+10.4f} '
      f'{(best_results["oa"] - unet_baseline_test_oa) / unet_baseline_test_oa * 100:>+9.1f}%')

Per-class comparison: vision-only vs best multimodal (hybrid_gemma3-4b)

Class          Baseline   Multimodal          Δ      Rel %
--------------------------------------------------------
Tree             0.5963       0.6148    +0.0185      +3.1%
Shrub            0.0848       0.1238    +0.0389     +45.9%
Grass            0.5842       0.6169    +0.0326      +5.6%
Crop             0.3948       0.4595    +0.0647     +16.4%
Built-up         0.2225       0.2718    +0.0493     +22.2%
Barren           0.2640       0.3171    +0.0531     +20.1%
Water            0.3098       0.3329    +0.0231      +7.4%
--------------------------------------------------------
mIoU             0.3509       0.3910    +0.0400     +11.4%
OA               0.8522       0.8871    +0.0349      +4.1%


In [35]:
# Persist multimodal test results alongside baseline for downstream analysis
multimodal_results_to_save = {
    'baseline_unetformer': {
        'test_miou':  unet_baseline_test_miou,
        'test_oa':    unet_baseline_test_oa,
        'class_ious': unet_baseline_class_ious,
    },
    'multimodal': {
        cap: {
            'test_miou':  res['miou'],
            'test_oa':    res['oa'],
            'test_loss':  res['loss'],
            'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']],
        }
        for cap, res in multimodal_test_results.items()
    },
    'class_names':      CLASS_NAMES,
    'best_caption_val': best_caption,        # from Section 6
    'best_caption_test': best_caption_test,  # might differ
}

results_path = RESULTS_DIR / 'multimodal_results.json'
with open(results_path, 'w') as f:
    json.dump(multimodal_results_to_save, f, indent=2)
print(f'Saved to {results_path}')

Saved to /content/drive/MyDrive/DI725_Project/results/multimodal_results.json


## 8. KL-divergence auxiliary loss ablation (Sub-question 2)

Take the best caption variant from the test evaluation (hybrid_gemma3-4b) and add a KL-divergence auxiliary loss between the predicted class distribution (spatial mean of softmax logits) and the per-image ground-truth class distribution from the captions CSV. The total loss becomes:

`total_loss = main_ce + 0.4 × aux_ce + kl_weight × kl(pred_dist || gt_dist)`

We sweep `kl_weight` over {0.1, 0.3, 0.5} to characterize how much explicit distribution supervision helps on top of text-derived guidance.

In [43]:
class RSMultiModalDatasetWithDist(Dataset):
    """Same as RSMultiModalDataset but additionally returns the CSV-derived
    ground-truth class distribution per image (used for the KL aux loss)."""

    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])

        # Ground-truth class distribution from CSV percentages (sum ≈ 1.0)
        gt_dist = torch.tensor(
            [row[c] / 100.0 for c in CLASS_NAMES],
            dtype=torch.float32,
        )

        return img, mask, caption, gt_dist


def make_loaders_with_dist(caption_col):
    train_dataset = RSMultiModalDatasetWithDist(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDatasetWithDist(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDatasetWithDist(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

In [44]:
def train_one_epoch_kl_csv(model, loader, optimizer, criterion, device, kl_weight):
    """Train one epoch with KL aux loss using CSV-derived gt_dist."""
    model.train()
    total_loss = 0
    for imgs, masks, captions, gt_dists in loader:
        imgs, masks, gt_dists = imgs.to(device), masks.to(device), gt_dists.to(device)
        logits_main, logits_aux = model(imgs, list(captions))

        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)

        # KL term: predicted class distribution vs CSV-derived ground truth
        pred_dist = F.softmax(logits_main, dim=1).mean(dim=[2, 3])
        kl_loss = F.kl_div(
            torch.log(pred_dist + 1e-8), gt_dists + 1e-8,
            reduction='batchmean',
        )
        loss = loss + kl_weight * kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_with_dist(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Evaluate; ignores the gt_dist tuple element (only used during training)."""
    model.eval()
    all_ious = [[] for _ in range(num_classes)]
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions, _ in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1).cpu()
        masks_cpu = masks.cpu()
        for pred, mask in zip(preds, masks_cpu):
            ious, _, _ = compute_metrics(pred, mask, num_classes)
            for c in range(num_classes):
                if not np.isnan(ious[c]):
                    all_ious[c].append(ious[c])
            total_correct += (pred == mask).sum().item()
            total_pixels  += mask.numel()
    class_ious = [np.mean(all_ious[c]) if all_ious[c] else float('nan') for c in range(num_classes)]
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [45]:
def train_multimodal_kl_csv(caption_col, kl_weight, num_epochs=NUM_EPOCHS, lr=LR,
                            run_suffix=None):
    """Train UNetFormerCLIP with KL aux loss using CSV-derived gt_dist."""
    base_name = caption_col.replace('-', '_').replace('/', '_')
    suffix = run_suffix or f'kl_csv_{int(kl_weight * 10):02d}'
    run_name = f'multimodal_{base_name}_{suffix}'
    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    train_loader, val_loader, _ = make_loaders_with_dist(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            'model':           'UNetFormerCLIP',
            'caption_col':     caption_col,
            'kl_weight':       kl_weight,
            'kl_source':       'csv',
            'num_epochs':      num_epochs,
            'lr':              lr,
            'weight_decay':    WEIGHT_DECAY,
            'batch_size':      BATCH_SIZE,
            'aux_loss_weight': AUX_LOSS_WEIGHT,
            'seed':            SEED,
        },
        reinit=True,
    )

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch_kl_csv(model, train_loader, optimizer, criterion, device, kl_weight)
        val_class_ious, val_miou, val_oa, val_loss = evaluate_with_dist(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()
    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path

In [46]:
# KL weight sweep — runs in sequence

kl_weights = [0.1, 0.3, 0.5]
kl_runs = {}

for w in kl_weights:
    suffix = f'kl_csv_{int(w * 10):02d}'
    print(f'\n{"=" * 60}')
    print(f'KL weight = {w}')
    print(f'{"=" * 60}\n')
    hist, miou, ckpt = train_multimodal_kl_csv(
        'hybrid_gemma3-4b', kl_weight=w, run_suffix=suffix
    )
    save_history(hist, f'multimodal_hybrid_gemma3_4b_{suffix}')
    kl_runs[w] = {'history': hist, 'best_val_miou': miou, 'ckpt': ckpt}

print('\n' + '=' * 60)
print('KL sweep complete')
print('=' * 60)
for w, run_info in kl_runs.items():
    print(f'  weight={w}: best val mIoU = {run_info["best_val_miou"]:.4f}')


KL weight = 0.1



Training multimodal_hybrid_gemma3_4b_kl_csv_01 for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.2065   0.4738   0.2469   0.7362   62.7s   BEST
    2   0.7707   0.3921   0.2754   0.7731   63.7s   BEST
    3   0.6953   0.3854   0.2699   0.7662   62.8s       
    4   0.6230   0.3839   0.2760   0.7981   63.0s   BEST
    5   0.6167   0.4065   0.2997   0.8032   62.9s   BEST
    6   0.5738   0.3245   0.3362   0.8295   62.9s   BEST
    7   0.5397   0.3135   0.2879   0.8183   62.7s       
    8   0.5074   0.3099   0.2916   0.7967   62.8s       
    9   0.4776   0.3263   0.3269   0.8120   63.0s       
   10   0.4596   0.2796   0.3348   0.8376   63.0s       
   11   0.4439   0.2836   0.3243   0.8258   63.4s       
   12   0.4342   0.2906   0.3590   0.8494   63.4s   BEST
   13   0.4281   0.3404   0.3302   0.8215   62.5s       
   14   0.4016   0.2629   0.3512   0.8488   63.6s       
   15   0.3826   0.2729  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▁▁▃▅▅▅▂▅▅▆▆▆▆▆▆▄▇▆▇▆█▇█▇▇███▇
val_iou/Built-up,▂▂▁▃▅▆▃▃▅▁▄▆▅▅▅▇▅▅▆███▆▇▇▇▇▆▇▇
val_iou/Crop,▁▂▃▂▂▄▄▄▄▅▅▅▄▆▆▆▇▆▆▇▇▆████████
val_iou/Grass,▁▂▃▃▄▆▄▄▅▆▄▇▄▆▆▆▇▆▇█▇▇▇███████
val_iou/Shrub,▁▄▂▅▁▄▄▃▁▆▄▄▃▅▄▄▅▆█▅▆▅▇▇▆▇▇▆█▇
val_iou/Tree,▁▄▂▄▃▅▄▄▄▅▅▆▅▇▅▇▆▇▇▇▇▇█▇██████
val_iou/Water,▃▅▄▁▅▇▁▄█▇▅█▇▆▆▄▆▆▅▅▆▆▆▆▇▆▇▆▆▇
+3,...



Best validation mIoU: 0.3862
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_kl_csv_01_history.json

KL weight = 0.3



Training multimodal_hybrid_gemma3_4b_kl_csv_03 for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.2698   0.5184   0.2405   0.7629   63.6s   BEST
    2   0.8340   0.3879   0.2885   0.8041   63.1s   BEST
    3   0.7325   0.3683   0.2991   0.8163   62.8s   BEST
    4   0.6563   0.4048   0.3059   0.8045   63.4s   BEST
    5   0.6153   0.3567   0.3277   0.8424   63.1s   BEST
    6   0.5877   0.3244   0.3000   0.8098   62.4s       
    7   0.5674   0.3717   0.2783   0.8148   61.6s       
    8   0.5310   0.3738   0.3276   0.8228   62.1s       
    9   0.5003   0.3026   0.3346   0.8362   63.0s   BEST
   10   0.4744   0.3263   0.3086   0.8242   62.4s       
   11   0.4598   0.3267   0.3101   0.8145   62.1s       
   12   0.4451   0.2729   0.3316   0.8480   61.3s       
   13   0.4673   0.2922   0.3528   0.8541   61.8s   BEST
   14   0.4076   0.2787   0.3193   0.8403   61.7s       
   15   0.3935   0.2593  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▄▃▃▅▄▄▅▅▅▅▆▇▅▇▆▆▅▅█▇▇▆█▇█▇▇▇▇
val_iou/Built-up,▁▅▃▅▅▄▃▇▆▅▇▅█▄▇▇▄▆▅▇▆█▆█▇▇▇▇█▆
val_iou/Crop,▁▃▃▂▃▃▁▄▄▄▅▅▅▄▆▇▇▆▆▆▇▇▇███████
val_iou/Grass,▁▃▃▃▆▄▄▅▅▃▃▆▅▆▇▆▆▆▇▇▇█▇▇██████
val_iou/Shrub,▅▃▄▁▅▄▂▆▄▂▁▄▆▃▇▇▇▅▆▆▇█▇▇▇██▆▇█
val_iou/Tree,▁▃▄▅▆▅▅▃▆▅▅▆▆▆▇▅▇█▇▇▇█████████
val_iou/Water,▂▃▆▆▄▃▁▆▅▄▃▄▅▃▇▅▄▆▆▇▇█▆▇▆▇▇▆█▇
+3,...



Best validation mIoU: 0.3932
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_kl_csv_03_history.json

KL weight = 0.5



Training multimodal_hybrid_gemma3_4b_kl_csv_05 for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3283   0.4621   0.2632   0.7546   63.3s   BEST
    2   0.8854   0.4361   0.2494   0.7658   62.9s       
    3   0.7445   0.3752   0.3148   0.8047   63.4s   BEST
    4   0.7242   0.3567   0.2954   0.8134   62.4s       
    5   0.6536   0.3562   0.3008   0.8197   62.7s       
    6   0.5941   0.4138   0.2739   0.7914   63.1s       
    7   0.5817   0.3348   0.3354   0.8399   63.9s   BEST
    8   0.5591   0.3611   0.3207   0.8352   63.1s       
    9   0.5615   0.3120   0.3404   0.8400   62.8s   BEST
   10   0.5298   0.3336   0.3002   0.8195   62.7s       
   11   0.4790   0.2881   0.3634   0.8606   63.6s   BEST
   12   0.4686   0.2703   0.3299   0.8341   63.3s       
   13   0.4514   0.2742   0.3527   0.8518   62.2s       
   14   0.4403   0.3672   0.3511   0.8279   63.0s       
   15   0.4179   0.2653  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▁▃▃▃▁▅▄▅▅▆▃▅▅▇▆▆▆▅▇▇▇▇█▇▇▇██▇
val_iou/Built-up,▁▄▅▄▅▃▆▇▇▄▇▂▄██▆▆▄▇█▇▆▇▇▇▇▇▇██
val_iou/Crop,▁▁▁▃▃▂▃▄▅▄▄▆▅▅▆▆▇▇▇▇▇█████████
val_iou/Grass,▁▁▄▄▃▄▅▅▅▅▆▅▆▆▇▇▆▇▇▇▇▇▇█▇█████
val_iou/Shrub,▁▄▂▅▂▅▄▃▆▂▄▄▄▄▃▆▄▅▄▂▆▇▅▇█▆█▇█▇
val_iou/Tree,▁▂▃▁▄▁▅▃▅▄▆▆▆▃▆▇▇▇▆▇█▇██▇█████
val_iou/Water,▆▁█▅▅▃▇▅▆▂▇▇██▇▇▆█▇▇█▇▇▇▇▇▇█▇█
+3,...



Best validation mIoU: 0.4010
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_kl_csv_05_history.json

KL sweep complete
  weight=0.1: best val mIoU = 0.3862
  weight=0.3: best val mIoU = 0.3932
  weight=0.5: best val mIoU = 0.4010


In [42]:
def evaluate_checkpoint_with_dist(caption_col, ckpt_path):
    """Evaluate a CSV-aware multimodal checkpoint on the test set."""
    _, _, test_loader = make_loaders_with_dist(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    class_ious, miou, oa, loss = evaluate_with_dist(model, test_loader, criterion, device)
    del model
    torch.cuda.empty_cache()
    return class_ious, miou, oa, loss


# Test eval for each KL run
for w, run_info in kl_runs.items():
    print(f'Evaluating KL weight = {w} on test set...')
    class_ious, test_miou, test_oa, test_loss = evaluate_checkpoint_with_dist(
        'hybrid_gemma3-4b', run_info['ckpt']
    )
    run_info['test_class_ious'] = class_ious
    run_info['test_miou']       = test_miou
    run_info['test_oa']         = test_oa
    run_info['test_loss']       = test_loss
    print(f'  test mIoU: {test_miou:.4f}, OA: {test_oa:.4f}')

Evaluating CSV-KL run on test set...


NameError: name 'ckpt_kl_csv' is not defined

In [ ]:
# Summary: KL weight vs test mIoU vs vision-only and best multimodal
best_no_kl = multimodal_test_results['hybrid_gemma3-4b']['miou']

print(f"{'KL weight':<12} {'Test mIoU':>10} {'Δ vs no-KL':>12} {'Δ vs baseline':>16}")
print('-' * 55)
print(f'{"vision-only":<12} {unet_baseline_test_miou:>10.4f} {"--":>12} {"--":>16}')
print(f'{"no KL":<12} {best_no_kl:>10.4f} {"--":>12} '
      f'{best_no_kl - unet_baseline_test_miou:>+16.4f}')
for w, run_info in kl_runs.items():
    delta_no_kl    = run_info['test_miou'] - best_no_kl
    delta_baseline = run_info['test_miou'] - unet_baseline_test_miou
    print(f'KL={w:<10} {run_info["test_miou"]:>10.4f} '
          f'{delta_no_kl:>+12.4f} {delta_baseline:>+16.4f}')

# Best KL configuration
best_kl_weight = max(kl_runs, key=lambda w: kl_runs[w]['test_miou'])
best_kl_test = kl_runs[best_kl_weight]['test_miou']
print(f'\nBest KL configuration: weight = {best_kl_weight} → test mIoU = {best_kl_test:.4f}')

In [ ]:
# Per-class comparison: vision-only vs best multimodal vs best KL config
best_kl_run = kl_runs[best_kl_weight]
best_no_kl_results = multimodal_test_results['hybrid_gemma3-4b']

print(f'\nPer-class: vision-only vs hybrid_gemma3-4b vs +KL (weight={best_kl_weight})\n')
print(f"{'Class':<11} {'Baseline':>10} {'Multimodal':>12} {'+KL':>10} {'Δ MM':>9} {'Δ KL':>9}")
print('-' * 67)
for i, name in enumerate(CLASS_NAMES):
    base = unet_baseline_class_ious[i]
    mm   = best_no_kl_results['class_ious'][i]
    kl   = best_kl_run['test_class_ious'][i]
    if base is None or np.isnan(mm) or np.isnan(kl):
        print(f'{name:<11} {"N/A":>10} {"N/A":>12} {"N/A":>10}')
        continue
    print(f'{name:<11} {base:>10.4f} {mm:>12.4f} {kl:>10.4f} '
          f'{mm-base:>+9.4f} {kl-mm:>+9.4f}')

print('-' * 67)
print(f'{"mIoU":<11} {unet_baseline_test_miou:>10.4f} {best_no_kl_results["miou"]:>12.4f} '
      f'{best_kl_run["test_miou"]:>10.4f} '
      f'{best_no_kl_results["miou"] - unet_baseline_test_miou:>+9.4f} '
      f'{best_kl_run["test_miou"] - best_no_kl_results["miou"]:>+9.4f}')
print(f'{"OA":<11} {unet_baseline_test_oa:>10.4f} {best_no_kl_results["oa"]:>12.4f} '
      f'{best_kl_run["test_oa"]:>10.4f} '
      f'{best_no_kl_results["oa"] - unet_baseline_test_oa:>+9.4f} '
      f'{best_kl_run["test_oa"] - best_no_kl_results["oa"]:>+9.4f}')

In [ ]:
# Update the multimodal results JSON with KL ablation
with open(RESULTS_DIR / 'multimodal_results.json') as f:
    results = json.load(f)

results['kl_ablation'] = {
    'caption_col': 'hybrid_gemma3-4b',
    'kl_source':   'csv',
    'runs': {
        f'kl_{w}': {
            'kl_weight':     w,
            'val_miou_best': float(run_info['best_val_miou']),
            'test_miou':     float(run_info['test_miou']),
            'test_oa':       float(run_info['test_oa']),
            'test_loss':     float(run_info['test_loss']),
            'class_ious':    [float(x) if not np.isnan(x) else None for x in run_info['test_class_ious']],
        }
        for w, run_info in kl_runs.items()
    },
    'best_kl_weight': float(best_kl_weight),
}

with open(RESULTS_DIR / 'multimodal_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Updated {RESULTS_DIR / "multimodal_results.json"}')